## Aim:
Test the usage of a chain of prompts within a single LLM for extracting CI_TYPE, damage and geolocation, as well as (cascading) impacts from disrupted CI .
Maybe the usage of a single model with a structured prompt chain is better then orchestrating two different LLMs for both tasks.

## Idea:
- 1st prompt use the one from `llm_geolocatoins.ipynb` for extracting CI_TYPE, its damage and location.
- 2nd prompt should be more specific, eg., based on first output extract further info about cascading impacts to other CI, economic and societal impacts

In [1]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# %env TORCH_CUDA_ARCH_LIST=8.6

# settings for distributed computing
%env WORLD_SIZE=1
%env RANK=0
%env LOCAL_RANK=0

# NOTE: # WORLD_SIZE: each GPU corresponds to one process (world = no. of processes within a group), processes communicate with each other enabling eg., distributed training
# NOTE: # RANK: IDs of the processes, ranging from 0 up to WORLD_SIZE - 1

env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0  # nvidia gpu
env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
env: WORLD_SIZE=1
env: RANK=0
env: LOCAL_RANK=0


In [2]:
import os
import sys
import re
import glob
import gc
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
from jinja2 import Template
from langchain_docling import DoclingLoader
import spacy
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GPTJForQuestionAnswering
)
import torch
import transformers


sys.path.append("../")
import src.settings as s

torch.manual_seed(42)

# set default location to store model before loading transformers
os.environ["HF_HOME"] = (
    "/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/"
)

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# %env MASTER_ADDR=127.0.0.1
# %env MASTER_PORT=6006

# # # Initialize distributed computing
# rank = int(os.environ["RANK"])
# device = torch.device(f"cuda:{rank}")
# torch.cuda.set_device(device)
# torch.distributed.init_process_group(backend="nccl")

In [4]:
pd.set_option(
    "display.max_colwidth", None
)  #  automatic linebreaks and multi-line cells.
pd.set_option("display.colheader_justify", "left")

## Generate CI_GEO-pairs

In [5]:
import importlib.util
# import spacy_transformers


try:
    nlp = spacy.load(s.settings.SPACY_MODEL)
except (OSError, ValueError) as e:
    print(f"spaCy language model '{s.settings.SPACY_MODEL}' not found. Downloading ...")
    ## loading transformer language model for NER requires additional package
    if (s.settings.SPACY_MODEL.endswith("_trf") and importlib.util.find_spec("spacy[transformers]") is None):
        !uv add spacy[transformers]
    !uv run python -m spacy download {s.settings.SPACY_MODEL}
    nlp = spacy.load(s.settings.SPACY_MODEL)

print(f"Loaded spaCy language model: {s.settings.SPACY_MODEL}")

Loaded spaCy language model: en_core_web_trf


In [6]:
## Create New entity for transport infrastructure and apply it on any doc

## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()


## call nlp model and create pipeline with new entity pattern
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


# store patterns in jsonl file, example:
# ruler.add_patterns([
#     {"label": "CI_TYPE", "pattern": "road?.+"},
#    {"label":"CI_TYPE","pattern":"rail.*$"},
# ])
# ruler.to_disk("../ner_patterns.jsonl")


## load doc
PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"
FILE_PATH = PARSED_TEXT_DIR + "Koks et al 2022 Brief communication_cleaned.md"
loader = DoclingLoader(FILE_PATH)  # use chunks from Docling.Loader
doc = loader.load()

## TODO make as pydantic class with fixed attributes
df_ci_geo = pd.DataFrame(
    columns=[
        "chunk_id",
        "ci_entity",
        "ci_entity_label",
        "geo_entity",
        "geo_entity_label",
        "token_distance",
    ]
)


## get most likely geolocation for each CI entity based on distance
for i, chunk in enumerate(doc):
    nlp_chunk = nlp(chunk.page_content)
    all_ents = [ent for ent in nlp_chunk.ents]
    ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
    ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
    fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

    # check if chunk contains CI_TYPE entities
    if len(ci_type_ents) > 0:
        print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
        print(
            f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
        )
        print(f"Chunk text [{i}]:", chunk.page_content)
        # print(f"{ {(ci_type_ents[i].text, ci_type_ents[i].label_) for i in range(len(ci_type_ents))} } ")

        # iterate over all entities within chunk
        for ent_idx in range(len(all_ents)):
            # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
            if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                ci_idx = ent_idx

                ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                distance_list = []
                idx_in_chunk = []
                try:
                    for ent_idx in range(len(all_ents)):
                        # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                        if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                            geo_idx = ent_idx
                            dist_ent_pair = np.abs(ci_idx - geo_idx)
                            distance_list.append(dist_ent_pair)
                            idx_in_chunk.append((ent_idx))
                            closest_pair_idx = np.argmin(
                                distance_list
                            )  # idx of closest GEO entity
                            distance_closest_pair = distance_list[closest_pair_idx]

                    threshold = 5  # max token distance between CI_TYPE and GEO entity
                    if distance_closest_pair > threshold:
                        print(
                            f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                        )
                        continue
                    else:
                        print(
                            f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                        )  # TODO constrain min.distance to max value (eg. 5 tokens), issue: likely when distance value is high that geolocation of Ci_type is mentioned in previous sentences or chunk

                    ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                    result_dict = {
                        "chunk_id": i,
                        "ci_entity": all_ents[ci_idx].text,
                        "ci_entity_label": all_ents[ci_idx].label_,
                        "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                        "geo_entity_label": all_ents[
                            idx_in_chunk[closest_pair_idx]
                        ].label_,
                        "token_distance": distance_closest_pair,
                    }
                    df_ci_geo = pd.concat(
                        [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                    )

                except IndexError:
                    print("No GEO entities found in this chunk.")
                    continue
                # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                # spacy.displacy.render(
                #     nlp_chunk, style="ent",
                #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                # )

        else:
            print("\nNo CI_TYPE or FAC entities found in this chunk.")
            continue

2026-01-05 15:20:20,568 - INFO - Going to convert document batch...
2026-01-05 15:20:20,568 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-05 15:20:20,569 - INFO - Processing document Koks et al 2022 Brief communication_cleaned.md
2026-01-05 15:20:20,616 - INFO - Finished converting document Koks et al 2022 Brief communication_cleaned.md in 0.05 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (556 > 512). Running this sequence through the model will result in indexing errors



Chunk [1], No. CI_TYPE and FAC entities: 4
Contains following entities for CI_TYPE: [bridges, sewage systems, schools, hospitals], FAC: []
Chunk text [1]: Abstract. Germany, Belgium and the Netherlands were hit by extreme precipitation and ﬂooding in July 2021. This brief communication provides an overview of the impacts to large-scale critical infrastructure systems and how recovery has progressed. The results show that Germany and Belgium were particularly affected, with many infrastructure assets severely damaged or completely destroyed. Impacts range from completely destroyed bridges and sewage systems, to severely damaged schools and hospitals. We ﬁnd that (large-scale) risk assessments, often focused on larger (river) ﬂood events, do not ﬁnd these local, but severe, impacts due to critical infrastructure failures. This may be the result of limited availability of validation material. As such, this brief communication not only will help to better understand how critical infrastru

In [7]:
df_ci_geo.loc[df_ci_geo["ci_entity"] == "rail"]  # .head(15)


,chunk_id,ci_entity,ci_entity_label,geo_entity,geo_entity_label,token_distance
17,6,rail,CI_TYPE,the Ahr valley,LOC,2
47,11,rail,CI_TYPE,Altenburg,GPE,4


In [8]:
doc[13].page_content

'We found no information regarding direct impact on solid-waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the affected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approximately 9 months as a temporary storage for debris (Couplez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from further upstream. Thousands of tonnes of tree debris (logs and\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\nE. E. Koks 

###  Prompt engineering

Jinja
- move template to separate file and load via get_template(), define conditions (e.g. user is technical or not) 
- use placeholders eg. for questions

Example code: 
* https://medium.com/@alecgg27895/jinja2-prompting-a-guide-on-using-jinja2-templates-for-prompt-management-in-genai-applications-e36e5c1243cf 
* https://newsletter.adaptiveengineer.com/p/why-jinja-rules-prompt-engineering

#### TODO make pydantic class model for expected JSON output



In [9]:
# ## left overs

### Line numbers  / citations
# Context:
# {% for item in context %}
# - {{ item.text }} (Line numbers: {{ item.line_numbers }})
# {% endfor %}
# In the field "line_numbers" provide the exact line numbers (in a list) from the context from which you extracted the impact information.
#     "line_numbers": "[...]"


## first output schema
# Return ONLY valid JSON in the following list format:
# [{
#     "infrastructure_type": "...",
#     "damage": "...",
#     "location": "...",
# }]

# Each nested dictionary describes one failure case.
# DO NOT add commentary or text outside the JSON.



In [10]:
from jinja2 import Environment, FileSystemLoader

## without s+e impacts
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."

# - 2nd prompt should be more specific, eg., based on first output extract further info about cascading impacts to other CI, economic and societal impacts
question_2 = "Based on the extracted failure cases from the first question, extract related information about cascading impacts to other infrastructure assets, economic and societal impacts."

env = Environment(loader=FileSystemLoader("../prompt_templates/"))
template = env.get_template("chain_of_prompts.txt")



In [11]:

# prompt_template = "  "
# template = Template(prompt_template)


## left overs

#  Try to be as specific as possible in your answer (bullet points), mention the impacts as numerical information along the location of the impact, and refer to the citations provided in the context.
# # Extract information about infrastructure failures based on the following question:

### first
#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

## Text_snippet column
#     In the field "text_snippet" provide the exact linenumbers (within a list) from the context which you used to extract the information about the infrastructure failure and its impacts.
# --> too long responses



## LLama

### Apply LLama on chunks 
* extract CI-GEO pairs (ie. most likely geolocation of each CI_TYPE ) (step 1)
* pass NER table to prompt for evaluating and improving LLM response (step 1)
* pass second Step to modle for extracting impact information based on the first step
* Test approach by applying it on three cleaned documents


In [12]:
# # empty CUDA cache
import gc
import torch

gc.collect()

torch.cuda.empty_cache()
# print(torch.cuda.memory_summary(device=None, abbreviated=False))

In [13]:
# init class for decoder and tokenizer


class DecoderModel:
    def __init__(self):
        login(
            token=os.environ["HUGGINGFACE_TOKEN"]
        )  # TODO replace by using pydantic settings

        # model_name = "google/gemma-3-4b-it" # "kallidavidson/TinyBERT_General_4L_312D"  # "huawei-noah/TinyBERT_General_4L_312D" # - for QA - less DWL
        model_name = "meta-llama/Llama-2-7b-chat-hf"
        # model_name = "EleutherAI/gpt-j-6B" #"distilbert-base-multilingual-cased"
        base_dir = "./huggingface_mirror"  # use default dir in .cache/
        model_dir = base_dir + "/hub/"  # + "models--" + model_name.replace("/", "--")
        print(model_dir)

        # quantization config
        # Load model with 4-bit quantization if applicable (use 4-bit integer instead of 32b floats) --> reduce the required VRAM for model application
        # see, https://huggingface.co/docs/transformers/quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        self.pipeline, self.tokenizer = self.initialize_model(
            model_name, model_dir, bnb_config
        )

    def initialize_model(self, model_name: str, model_dir: str = None, bnb_config=None):
        # Model and Tokenizer initialization
        if not os.path.exists(model_dir):
            print("Model directory not found. Downloading model...")
            os.makedirs(model_dir, exist_ok=True)

            device = transformers.infer_device()
            print(f"Using device: {device}")
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype="auto",
                attn_implementation="flash_attention_2",  # use with 4-bit quantization,
                # --> flash attention enables to use much larger sequence lengths without running into OOM issues
                quantization_config=bnb_config,
                # max_memory={0: "2GB", 1: "10GB"},  # distribute memory across GPUs
            )
            model.save_pretrained(model_dir)
            tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            tokenizer.save_pretrained(model_dir)

            print("Downloaded model and tokenizer")

        else:
            print(f"Using locally saved model from {model_dir}")

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                cache_dir=model_dir,
                local_files_only=True,  # tp_plan="auto" # set tensor parallel model (ie. splits model on multiple GPU)
                # dtype="auto",
                attn_implementation="flash_attention_2",  # use with 4-bit quantization,
                # --> flash attention enables to use much larger sequence lengths without running into OOM issues
                quantization_config=bnb_config,
                # tp_plan="auto",  # automatically use a tensor parallelism plan based on predefined configuration of the model (i.e. partition model on both GPUs)
            )
            # print("Tensor parallel plan:", model._tp_plan)

            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                use_fast=True,
                cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
            )

        # reduce further memory usage
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.use_checkpointing = True

        torch.cuda.empty_cache()

        # Pipeline setup for question answering
        pipeline = transformers.pipeline(  # load model locally from wsl .cache\
            "text-generation",
            # "question-answering",  # task defining which pipeline is returned
            model=model,
            tokenizer=tokenizer,
            # (return_tensors="pt"),  # load specific tokenizer based on model-name (via AutoTokenizer) ensuring text is tokenized in accordance to the way the model was trained
            max_new_tokens=1024, # high max toke otherwise output is truncated
            device_map="auto",
        )
        return pipeline, tokenizer

    def generate_response(
        self, question_1: str, question_2: str, context: list, chunk_id: int
    ):
        # NOTE context includes .text (document chunk) and .ci_locations (geo-ci entities)
    
        rendered_prompt = template.render(
            context=context,
            question_1=question_1,
            question_2=question_2,
        )
        print(f"Generating response for chunk_id: {chunk_id} ...")

        sequences = self.pipeline(
            rendered_prompt,  # jinja template
            max_new_tokens=1024, # use default to not truncate the LLM response
            do_sample=True,
            num_beams=1,  # select token based on probability distribution over entire model’s vocabulary
            # top_k=10,
            # top_p=0.5,
            temperature=0.2,
            # num_return_sequences=1,
            eos_token_id=self.tokenizer.eos_token_id,
            return_full_text=False,  # allow bullet point answers
        )
        # Extracting and returning the generated text
        return sequences

In [14]:

PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"


## call language model for recognition of CI and geolocation
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


df_responses = pd.DataFrame(
    columns=[
        "chunk_id",
        "infrastructure_type",
        "damage",
        "location",
        "impacts_to_other_infrastructure_assets",
        "societal_impact",
        "economic_impact",
        "line_numbers",
    ]
)

## init LLM pipeline
decoder_model = DecoderModel()



## iterate over documents
for i, filename in enumerate(glob.glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md")))):

    no_documents = len(glob.glob(str(Path(PARSED_TEXT_DIR, "*cleaned.md"))))
    filepath = Path(filename)
    filename_stem = filepath.stem


    print(f"\n\n ######## -------- Processing document [{i+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
    try:
        authors, year, title = re.findall(citation_pattern, filename_stem)[0]
        citation = f"{authors} {year}"
    except AttributeError as e:
        print(f"Could not extract citation from title: {e}")
        citation = filename_stem


    print(f"\n ######## -------- Getting geolocation of CI assets -------- ######## \n")
    ## Create New entity for transport infrastructure and apply it on any doc

    ## TODO make as pydantic class with fixed attributes
    df_ci_geo = pd.DataFrame(
        columns=[
            "chunk_id",
            "ci_entity",
            "ci_entity_label",
            "geo_entity",
            "geo_entity_label",
            "token_distance",
        ]
    )

    ## load doc
    loader = DoclingLoader(filepath)  # use chunks from Docling.Loader
    doc = loader.load()
    doc = doc[5:15]

    ## get most likely geolocation for each CI entity based on distance
    for i, chunk in enumerate(doc):
        nlp_chunk = nlp(chunk.page_content)
        all_ents = [ent for ent in nlp_chunk.ents]
        ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
        ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
        fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

        # check if chunk contains CI_TYPE entities
        if len(ci_type_ents) > 0:
            print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
            # print(
            #     f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
            # )
            # print(f"Chunk text [{i}]:", chunk.page_content)

            # iterate over all entities within chunk
            for ent_idx in range(len(all_ents)):
                # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
                if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                    ci_idx = ent_idx

                    ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                    distance_list = []
                    idx_in_chunk = []
                    try:
                        for ent_idx in range(len(all_ents)):
                            # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                            if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                                geo_idx = ent_idx
                                dist_ent_pair = np.abs(ci_idx - geo_idx)
                                distance_list.append(dist_ent_pair)
                                idx_in_chunk.append((ent_idx))
                                closest_pair_idx = np.argmin(
                                    distance_list
                                )  # idx of closest GEO entity
                                distance_closest_pair = distance_list[closest_pair_idx]

                        threshold = 5  # max token distance between CI_TYPE and GEO entity
                        if distance_closest_pair > threshold:
                            print(
                                f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                            )
                            continue
                        else:
                            pass
                            # print(
                            #     f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                            # )

                        ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                        result_dict = {
                            "chunk_id": i,
                            "ci_entity": all_ents[ci_idx].text,
                            "ci_entity_label": all_ents[ci_idx].label_,
                            "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                            "geo_entity_label": all_ents[
                                idx_in_chunk[closest_pair_idx]
                            ].label_,
                            "token_distance": distance_closest_pair,
                        }
                        df_ci_geo = pd.concat(
                            [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                        )

                    except IndexError:
                        # print("No GEO entities found in this chunk.")
                        continue
                    # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                    # spacy.displacy.render(
                    #     nlp_chunk, style="ent",
                    #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                    # )

            else:
                print("\nNo CI_TYPE or FAC entities found in this chunk.")
                continue


    print(f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n")

    ## apply decoder on each chunk
    ## TODO replace iteration by loading entire document and use recursive chunking from langchain
    for j, chunk in enumerate(doc):

        if df_ci_geo.loc[df_ci_geo["chunk_id"] == j].empty:
            continue

        context = [
            {
                "text": chunk.page_content,
                "citation": citation,
                "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"] == j],
            },  # TODO use author names or Primary keys from DB later
        ]

        response = decoder_model.generate_response(
            question_1=question_1, 
            question_2=question_2,
            context=context, 
            chunk_id=j
        )

        ## postprocess response
        resp = response[0]["generated_text"].replace("\n", "")
        try:
            resp = (resp.split("]")[0] + "]")  # remove potential text outside of json object
            df_resp = pd.read_json(StringIO(resp))
            df_resp["chunk_id"] = j  # add chunk id as identifier
            df_resp["citation"] = citation  # add citation info
            df_responses = pd.concat([df_responses, df_resp], ignore_index=True)
        except ValueError as e:
            print(f"Cannot add response to output dataframe: {e}, \n{resp}")



    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()

SpanRuler already exists in pipeline.
./huggingface_mirror/hub/
Using locally saved model from ./huggingface_mirror/hub/


Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.02s/it]
Device set to use cuda:0




 ######## -------- Processing document [1/3]: Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md -------- ######## 


 ######## -------- Getting geolocation of CI assets -------- ######## 



2026-01-05 15:20:43,977 - INFO - Going to convert document batch...
2026-01-05 15:20:43,977 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-05 15:20:43,978 - INFO - Processing document Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md
2026-01-05 15:20:44,156 - INFO - Finished converting document Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md in 0.18 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 512). Running this sequence through the model will result in indexing errors



Chunk [0], No. CI_TYPE and FAC entities: 4

No CI_TYPE or FAC entities found in this chunk.

Chunk [3], No. CI_TYPE and FAC entities: 2

No CI_TYPE or FAC entities found in this chunk.

Chunk [4], No. CI_TYPE and FAC entities: 2

No CI_TYPE or FAC entities found in this chunk.

Chunk [6], No. CI_TYPE and FAC entities: 2

No CI_TYPE or FAC entities found in this chunk.

Chunk [7], No. CI_TYPE and FAC entities: 1

No CI_TYPE or FAC entities found in this chunk.

  #############  -------- Text-2-Data: Mohr 2022 A multi-disciplinary analysis of the exceptional flood event of July 2021 in central Europe - Part 1 Event desciption and analysis_cleaned.md -------- #############  

Generating response for chunk_id: 0 ...
Generating response for chunk_id: 7 ...
Cannot add response to output dataframe: Trailing data, 
{"infrastructure_type": "reservoirs","damage": "flooding","location": "Ahr","impacts_to_other_infrastructure_assets": "NAN","societal_impact": "NAN","economic_impact": "NAN"}{"infr

...


Chunk [0], No. CI_TYPE and FAC entities: 10
 Token distance between CI_TYPE/FAR and next GEO entity is 14 and thus larger than the allowed distance of 5 tokens 

No CI_TYPE or FAC entities found in this chunk.

Chunk [1], No. CI_TYPE and FAC entities: 7

No CI_TYPE or FAC entities found in this chunk.

Chunk [2], No. CI_TYPE and FAC entities: 5

No CI_TYPE or FAC entities found in this chunk.

Chunk [3], No. CI_TYPE and FAC entities: 4

No CI_TYPE or FAC entities found in this chunk.

Chunk [4], No. CI_TYPE and FAC entities: 2

No CI_TYPE or FAC entities found in this chunk.

Chunk [5], No. CI_TYPE and FAC entities: 4

No CI_TYPE or FAC entities found in this chunk.

Chunk [6], No. CI_TYPE and FAC entities: 12

No CI_TYPE or FAC entities found in this chunk.

Chunk [7], No. CI_TYPE and FAC entities: 5

No CI_TYPE or FAC entities found in this chunk.

Chunk [8], No. CI_TYPE and FAC entities: 7

No CI_TYPE or FAC entities found in this chunk.

Chunk [9], No. CI_TYPE and FAC entities: 9


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generating response for chunk_id: 8 ...
Generating response for chunk_id: 9 ...


 ######## -------- Processing document [3/3]: Korzilius 2021 Nach der Flut_cleaned.md -------- ######## 


 ######## -------- Getting geolocation of CI assets -------- ######## 



2026-01-05 15:34:33,621 - INFO - Going to convert document batch...
2026-01-05 15:34:33,622 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-05 15:34:33,623 - INFO - Processing document Korzilius 2021 Nach der Flut_cleaned.md
2026-01-05 15:34:33,744 - INFO - Finished converting document Korzilius 2021 Nach der Flut_cleaned.md in 0.13 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors



Chunk [0], No. CI_TYPE and FAC entities: 1

No CI_TYPE or FAC entities found in this chunk.

Chunk [9], No. CI_TYPE and FAC entities: 1

No CI_TYPE or FAC entities found in this chunk.

  #############  -------- Text-2-Data: Korzilius 2021 Nach der Flut_cleaned.md -------- #############  

Generating response for chunk_id: 9 ...
Cannot add response to output dataframe: Trailing data, 
 {"infrastructure_type": "Hospital","damage": "Zerstörung","location": "Erftstadt","impacts_to_other_infrastructure_assets": "NAN","societal_impact": "NAN","economic_impact": "NAN"}]


In [ ]:
import json
r = json.dumps(response[0])
# json.loads(r)
print(json.dumps(response[0],  indent=4)) 


# TODO test without first JSON


#response[0]#["generated_text"].replace("\n", "")

In [25]:
df_responses

,chunk_id,infrastructure_type,damage,location,impacts_to_other_infrastructure_assets,societal_impact,economic_impact,line_numbers,citation
0,0,roads,severe,Rhine,Cascading impacts to railways and bridges,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
1,0,bridges,severe,Rhine,Cascading impacts to railways and roads,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
2,0,railways,severe,Rhine,Cascading impacts to roads and bridges,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
3,0,road,severely damaged,Germany,NaN,NaN,NaN,NaN,Koks et al 2022
4,0,railway,severely damaged,Germany,NaN,NaN,NaN,NaN,Koks et al 2022
5,0,bridges,destroyed,the Ahr valley,NaN,NaN,NaN,NaN,Koks et al 2022
6,0,motorways,closed,the Ahr valley,NaN,NaN,NaN,NaN,Koks et al 2022
7,2,road,limited,Maastricht,NaN,NaN,NaN,NaN,Koks et al 2022
8,2,railway,several railway sections were closed,Maastricht and Liége,NaN,NaN,NaN,NaN,Koks et al 2022
9,3,Electricity infrastructure,"Severe damage to power lines, transformers, and distribution stations",Germany,"Cascading impacts to other infrastructure assets, such as communication and transportation systems","Disruption of essential services, including healthcare and emergency services",Estimated economic losses of €100 million,NaN,Koks et al 2022


#### response

In [ ]:
safety_df = df_responses.copy()


In [ ]:

# save to disk along with prompt text


OUTPUT_DIR = "../" + s.settings.PATH_DATA + "llm_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outfile_name = "responses_chain_of_prompts_llama2.csv"
outfile_path = OUTPUT_DIR +  outfile_name


if not os.path.isfile(outfile_path):
    outfile_response_path = Path(outfile_path)
    outfile_prompt_path = Path(OUTPUT_DIR +  "prompt_" + outfile_name.replace(".csv", ".txt"))

    print(f"Saving prompt and LLM response to {outfile_response_path} ...")
    with open(outfile_prompt_path, "w") as f:
        f.write(template.render())
    df_responses.to_csv(outfile_response_path, index=False)
else:
    print(f"WARNING: Output file {Path(outfile_name).stem} already exists. Skip saving to avoid overwriting ...")



Saving prompt and LLM response to ../../data/llm_outputs ...


In [31]:
df_responses


,chunk_id,infrastructure_type,damage,location,impacts_to_other_infrastructure_assets,societal_impact,economic_impact,line_numbers,citation
0,0,roads,severe,Rhine,Cascading impacts to railways and bridges,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
1,0,bridges,severe,Rhine,Cascading impacts to railways and roads,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
2,0,railways,severe,Rhine,Cascading impacts to roads and bridges,"Disruption of transportation and supply chains, increased travel times and costs, and potential loss of life",Estimated losses of EUR 33 billion,NaN,Mohr 2022
3,0,road,severely damaged,Germany,NaN,NaN,NaN,NaN,Koks et al 2022
4,0,railway,severely damaged,Germany,NaN,NaN,NaN,NaN,Koks et al 2022
5,0,bridges,destroyed,the Ahr valley,NaN,NaN,NaN,NaN,Koks et al 2022
6,0,motorways,closed,the Ahr valley,NaN,NaN,NaN,NaN,Koks et al 2022
7,2,road,limited,Maastricht,NaN,NaN,NaN,NaN,Koks et al 2022
8,2,railway,several railway sections were closed,Maastricht and Liége,NaN,NaN,NaN,NaN,Koks et al 2022
9,3,Electricity infrastructure,"Severe damage to power lines, transformers, and distribution stations",Germany,"Cascading impacts to other infrastructure assets, such as communication and transportation systems","Disruption of essential services, including healthcare and emergency services",Estimated economic losses of €100 million,NaN,Koks et al 2022


# Evaluation

### Manual comparison CI_location_table vs LLm response


#### chunk 5
In Germany, road and railway infrastructure was severely damaged as documented exemplarily in Fig. 1. Cost estimates reach up to EURO 2 billion Euro (MDR, 2021). More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR 100 million (Hauser, 2021). Of the 112 bridges in the ﬂooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the ﬂood event (MDR, 2021). Over 74 km of roads, paths and bridges in the Ahr valley have been (critically) damaged. In some cases, repairs are expected to take months to years (Zeit Online, 2021). For example, major freeway sections, including parts of the A1 motorway, were closed until early 2022 (24Rhein, 2022). In addition, about 50 000 cars were damaged, causing insurance claims of some EUR 450 million (ADAC, 2021). The German railway provider Deutsche Bahn expects asset damages of around EUR 1.3 billion. Among other things, 180 level crossings, almost 40 signal'

In [ ]:
doc[5].page_content

In [ ]:
df_ci_geo[df_ci_geo["chunk_id"] == 5]

In [ ]:
df_responses[df_responses["chunk_id"] == 5]

In [ ]:
print(
    f"CI_TYPE \n LLM responses:\n {df_responses.infrastructure_type.unique()}, \n\n NER pairs:\n {df_ci_geo.ci_entity.unique()}"
)

In [ ]:
print(
    f"LOCATION:\n LLm responses:\n {df_responses.location.unique()}, \n\n NER pairs:\n {df_ci_geo.geo_entity.unique()}"
)

#### chunk 13 +14
'We found no information regarding direct impact on solid-waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the affected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approximately 9 months as a temporary storage for debris (Couplez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from further upstream. Thousands of tonnes of tree debris (logs and\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\nE. E. Koks et al.: Flood impacts to infrastructure'


'of running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million. Within 3.5 weeks, the hospital was partly operational, and within 3 months, all hospital operations continued normally (SAH Eschweiler, 2021). The Mutterhaus Ehrang hospital in Trier (Germany) is now permanently closed as the hospital is too severely damaged to rebuild. Furthermore, in the region of Rhineland-Palatinate (Germany), 19'


In [ ]:
doc[14].page_content

In [ ]:
df_ci_geo[df_ci_geo["chunk_id"].isin([13, 14])]

In [ ]:
df_responses[df_responses["chunk_id"].isin([13, 14])]

In [ ]:
print(
    f"CI_TYPE \n LLM responses:\n {df_responses.infrastructure_type.unique()}, \n\n NER pairs:\n {df_ci_geo.ci_entity.unique()}"
)

In [ ]:
print(
    f"LOCATION:\n LLm responses:\n {df_responses.location.unique()}, \n\n NER pairs:\n {df_ci_geo.geo_entity.unique()}"
)

## Semantic Textual Similarity (STS)

Calculating the STS for both model configurations (chain of prompts, orchestration of models)
The outputs are cosine similarity scores for similar model outputs per chunk. They are ranked by score for each model, restricted to the top 20 results.  


### Chain of prompts vs domain-expertise 

In [ ]:
import io
from pathlib import Path
import config
import pickle
import time
import warnings
import pandas as pd
import torch
from transformers.models.bert.modeling_bert import BertModel
from transformers import AutoModel
# from utils.training import topic_search, topic_search_lm


## Configuration
similarity_threshold = 0.75

#  Define the models and their names for easier naming of files
lm_dict = {'chain_of_prompts': "llama_2", 'orchestrated_llms': "llama_2"}

#  Define if you want to skip calculations using pretrained models
use_pretrained = True

#  How many tags of the models should be considered for the results
n_rows = 25
## STS calculation and results creation


#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)

#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
PATH_EVAL_DATA = Path('../data/evaluation/manual_extracted')
df_eval = pd.read_csv(PATH_EVAL_DATA / 'table_ci_impacts_sm.csv')

#  Define folder for handling and writing outputs
def write_to_file(data, out_folder, filename):
    """Convert output to DataFrame and write to file"""
    df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
    #  Sort the DataFrame by similarity (explicitly)
    df = df.sort_values(by='sts_score', ascending=False)
    #  Assign integers to ranking
    df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
    #  Only keep the first 20 resulting tags
    df = df.head(50)
    #  Save to file
    df.to_csv(out_folder / f'{filename}_output.csv', index=False)

#  Fill run metrics to dictionary
def handle_metrics(metrics, model_name, length, end_time, start_time):
    print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
    metrics.append({
        'modelname': model_name,
        'runtime': round(end_time - start_time, 2),
        'tagcount': length
    })
    return metrics

class CPU_Unpickler(pickle.Unpickler):
    """Fix for having issues with loading models on CPU"""
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else: return super().find_class(module, name)

#  Load the tags with the embeddings
tags = pd.read_csv(PATH_EVAL_DATA / 'tags.csv', usecols=['key', 'value'])
tags = [f"{key}={value}" for key, value in zip(tags['key'], tags['value'])]

#  Iterate over the topics and calculate the STS for each model
for topic in topics:
    metrics = []
    print(f'Calculating STS for topic/tag {topic}:')
    print('--------------------------------------')

    #  Create output folder for each topic
    out_name = topic.replace('=', '_')
    out_folder = Path(city) / 'output' / out_name
    out_folder.mkdir(exist_ok=True, parents=True)

    #  Custom models (TinyBert & Bert)
#    for lm in lm_dict.keys():
    for lm in ["tiny"]:
        print(f'Custom {lm_dict[lm].title()} model...')
        model_path = city + f'/Model/model_bert_{lm}.pkl'
        start_time = time.time()
        with open(model_path, 'rb') as f:
            if torch.cuda.is_available():
                model = pickle.load(f)
            else:
                model = CPU_Unpickler(f).load()
        output = topic_search(model, lm, topic, tags, similarity_threshold)
        end_time = time.time()
        model_name = f'custom_{lm_dict[lm].lower()}'
        write_to_file(output, out_folder, model_name)
        metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

    #  Pre-trained models
    if use_pretrained:
        #  Pre-trained TinyBert
        lm = lm_dict['tiny']
        print(f'Pre-trained {lm.title()} model...')
        start_time = time.time()
        model = AutoModel.from_pretrained(config.lm_names[lm])
        output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
        end_time = time.time()
        model_name = f'pretrained_{lm.lower()}'
        write_to_file(output, out_folder, model_name)
        metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

        #  Pre-trained Bert
        lm = lm_dict['large']
        print(f'Pre-trained {lm.title()} model...')
        start_time = time.time()
        model = BertModel.from_pretrained(config.lm_names[lm])
        output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
        end_time = time.time()
        model_name = f'pretrained_{lm.lower()}'
        write_to_file(output, out_folder, model_name)
        metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

    print('\n')
    df = pd.DataFrame(metrics)
    df.to_csv(out_folder / 'metrics.csv', index=False)
    
print('Done!')
## Merge all topic outputs and combine with reference file to compare performance
for topic in topics:
    print(f'Handling outputs for topic/tag {topic}...')
    topic_name = topic.replace('=', '_')

    in_dir = Path(city) / 'output' / topic_name
    n_rows = 25 #  How many tags of the models should be considered

    #  Load the output files
    custom_tinybert = pd.read_csv(in_dir / 'custom_tinybert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    custom_bert = pd.read_csv(in_dir / 'custom_bert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    pretrained_tinybert = pd.read_csv(in_dir / 'pretrained_tinybert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    pretrained_bert = pd.read_csv(in_dir / 'pretrained_bert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)

    #  Load references
    reference = pd.read_csv(PATH_EVAL_DATA / f'{topic_name}.csv', usecols=['tag']).set_index('tag')

    #  Align output files with references (keep only references)
    reference, custom_tinybert = reference.align(custom_tinybert, join='left', axis=0)
    reference, custom_bert = reference.align(custom_bert, join='left', axis=0)
    reference, pretrained_tinybert = reference.align(pretrained_tinybert, join='left', axis=0)
    reference, pretrained_bert = reference.align(pretrained_bert, join='left', axis=0)

    #  Merge all aligned dataframes and rename cols to models
    merged_df = pd.concat([reference, custom_tinybert, custom_bert, pretrained_tinybert, pretrained_bert], axis=1)
    merged_df.columns = ['custom_tinybert', 'custom_bert', 'pretrained_tinybert', 'pretrained_bert']

    #  Add empty row for separation
    merged_df.loc[''] = None

    #  Calculate the weighted harmonic sum with weights based on reference tag importance/rank
    weights = 1 / (pd.Series(range(1, len(merged_df) + 1), index=merged_df.index))
    weighted_harmonic_sum = merged_df.apply(lambda col: round((weights / col).sum(), 3), axis=0)
    merged_df.loc['Weighted Harmonic Sum'] = weighted_harmonic_sum

    #  Add true positives count to results
    counts = merged_df[:-1].notna().sum()
    new_row = {col: counts[col] for col in ['custom_tinybert', 'custom_bert', 'pretrained_tinybert', 'pretrained_bert']}
    merged_df.loc['True Positives'] = new_row

    #  Add run metrics to results
    metrics = pd.read_csv(in_dir / 'metrics.csv').set_index('modelname')
    merged_df.loc['Runtime [s]'] = metrics['runtime']
    merged_df.loc['Number of Tags'] = metrics['tagcount']

    #  File type casts
    merged_df
    #  Save to file
    out_file = in_dir / 'results.csv'
    merged_df.to_csv(out_file)
    print(f'-> Saved merged output to {out_file}.')

print('\nDone!')


FileNotFoundError: [Errno 2] No such file or directory: '../data/evaluation/manual_extracted/table_ci_impacts_sm.csv'

### LangExtract vs chain of prompts

## Test alternative approaches for entity linking / relation extraction

In [ ]:
import torch
import gc

print(torch.cuda.memory_summary(device=None, abbreviated=False))
# # empyty CUDA cache
gc.collect()

torch.cuda.empty_cache()
# print(torch.cuda.memory_summary(device=None, abbreviated=False))